In [202]:
import pandas as pd

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (mean_absolute_error, root_mean_squared_error, r2_score)
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_regression, chi2

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [203]:
df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("Test.csv")
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [204]:
print(df_train.head())
print("***************************************************")
print(df_train.info())
print("***************************************************")
print(df_train.describe())

   Id  MSSubClass MSZoning  LotFrontage  LotArea Street Alley LotShape  \
0   1          60       RL         65.0     8450   Pave   NaN      Reg   
1   2          20       RL         80.0     9600   Pave   NaN      Reg   
2   3          60       RL         68.0    11250   Pave   NaN      IR1   
3   4          70       RL         60.0     9550   Pave   NaN      IR1   
4   5          60       RL         84.0    14260   Pave   NaN      IR1   

  LandContour Utilities LotConfig LandSlope Neighborhood Condition1  \
0         Lvl    AllPub    Inside       Gtl      CollgCr       Norm   
1         Lvl    AllPub       FR2       Gtl      Veenker      Feedr   
2         Lvl    AllPub    Inside       Gtl      CollgCr       Norm   
3         Lvl    AllPub    Corner       Gtl      Crawfor       Norm   
4         Lvl    AllPub       FR2       Gtl      NoRidge       Norm   

  Condition2 BldgType HouseStyle  OverallQual  OverallCond  YearBuilt  \
0       Norm     1Fam     2Story            7          

In [205]:
print("Is Null:")
fields_with_null_values = df_train.isnull().sum()
fields_with_null_values = fields_with_null_values[fields_with_null_values > 0]

null_info = pd.DataFrame({
    'Missing Values': fields_with_null_values,
    'Data Type': df_train.dtypes[fields_with_null_values.index]
})

print(null_info)

Is Null:
              Missing Values Data Type
LotFrontage              259   float64
Alley                   1369       str
MasVnrType               872       str
MasVnrArea                 8   float64
BsmtQual                  37       str
BsmtCond                  37       str
BsmtExposure              38       str
BsmtFinType1              37       str
BsmtFinType2              38       str
Electrical                 1       str
FireplaceQu              690       str
GarageType                81       str
GarageYrBlt               81   float64
GarageFinish              81       str
GarageQual                81       str
GarageCond                81       str
PoolQC                  1453       str
Fence                   1179       str
MiscFeature             1406       str


In [206]:
df_train["LotFrontage"] = (
    df_train.groupby("Neighborhood")["LotFrontage"]
      .transform(lambda x: x.fillna(x.median()))
)

df_train["Electrical"] = df_train["Electrical"].fillna(
    df_train["Electrical"].mode()[0]
)

df_train["Alley"] = df_train["Alley"].fillna("None")
df_train["MasVnrType"] = df_train["MasVnrType"].fillna("None")
df_train["MasVnrArea"] = df_train["MasVnrArea"].fillna(0)
df_train["BsmtQual"] = df_train["BsmtQual"].fillna("None")
df_train["BsmtCond"] = df_train["BsmtCond"].fillna("None")
df_train["BsmtExposure"] = df_train["BsmtExposure"].fillna("None")
df_train["BsmtFinType1"] = df_train["BsmtFinType1"].fillna("None")
df_train["BsmtFinType2"] = df_train["BsmtFinType2"].fillna("None")
df_train["FireplaceQu"] = df_train["FireplaceQu"].fillna("None")
df_train["GarageType"] = df_train["GarageType"].fillna("None")
df_train["GarageYrBlt"] = df_train["GarageYrBlt"].fillna(0)
df_train["GarageFinish"] = df_train["GarageFinish"].fillna("None")
df_train["GarageQual"] = df_train["GarageQual"].fillna("None")
df_train["GarageCond"] = df_train["GarageCond"].fillna("None")
df_train["PoolQC"] = df_train["PoolQC"].fillna("None")
df_train["Fence"] = df_train["Fence"].fillna("None")
df_train["MiscFeature"] = df_train["MiscFeature"].fillna("None")

In [207]:
print("Is Null:")
fields_with_null_values = df_test.isnull().sum()
fields_with_null_values = fields_with_null_values[fields_with_null_values > 0]

null_info = pd.DataFrame({
    'Missing Values': fields_with_null_values,
    'Data Type': df_test.dtypes[fields_with_null_values.index]
})

print(null_info)

Is Null:
              Missing Values Data Type
MSZoning                   4       str
LotFrontage              227   float64
Alley                   1352       str
Utilities                  2       str
Exterior1st                1       str
Exterior2nd                1       str
MasVnrType               894       str
MasVnrArea                15   float64
BsmtQual                  44       str
BsmtCond                  45       str
BsmtExposure              44       str
BsmtFinType1              42       str
BsmtFinSF1                 1   float64
BsmtFinType2              42       str
BsmtFinSF2                 1   float64
BsmtUnfSF                  1   float64
TotalBsmtSF                1   float64
BsmtFullBath               2   float64
BsmtHalfBath               2   float64
KitchenQual                1       str
Functional                 2       str
FireplaceQu              730       str
GarageType                76       str
GarageYrBlt               78   float64
GarageFinish    

In [208]:
neighborhood_avg = [
  "LotFrontage"
]

str_category = [
  "MSZoning",
  "Utilities",
  "Exterior1st",
  "Exterior2nd",
  "SaleType"
]

na_to_none = [
  "Alley",
  "MasVnrType",
  "BsmtQual",
  "BsmtCond",
  "BsmtExposure",
  "BsmtFinType1",
  "BsmtFinType2",
  "KitchenQual",
  "Functional",
  "FireplaceQu",
  "GarageType",
  "GarageFinish",
  "GarageQual",
  "GarageCond",
  "PoolQC",
  "Fence",
  "MiscFeature"
]

na_to_0 = [
  "BsmtFinSF1",
  "BsmtFinSF2",
  "BsmtUnfSF",
  "TotalBsmtSF",
  "BsmtFullBath",
  "BsmtHalfBath",
  "MasVnrArea",
  "GarageYrBlt",
  "GarageCars",
  "GarageArea"
]

for item in neighborhood_avg:
  df_test[item] = (
    df_test.groupby("Neighborhood")[item]
    .transform(lambda x: x.fillna(x.median()))
  )

for item in str_category:
  df_test[item] = df_test[item].fillna(
      df_test[item].mode()[0]
  )

for item in na_to_none:
  df_test[item] = df_test[item].fillna("None")

for item in na_to_0:
  df_test[item] = df_test[item].fillna(0)

In [209]:
# Creating Classification Dataset
classification_df_train = df_train.copy()

# 0 = Low, 1 = Medium, 2 = High
classification_df_train["PriceCategory"] = pd.qcut(
    classification_df_train["SalePrice"],
    q=3,
    labels=[0, 1, 2]
)

classification_df_train.drop(
    columns=["SalePrice"],
    inplace=True
)
classification_df_train["PriceCategory"].value_counts()


PriceCategory
1    490
0    487
2    483
Name: count, dtype: int64

Nominal Catagories:
"MSSubClass",
"MSZoning",
"Street",
"Alley",
"LotShape",
"LandContour",
"Utilities",
"LotConfig",
"LandSlope",
"Neighborhood",
"Condition1",
"Condition2",
"BldgType",
"HouseStyle",
"RoofStyle",
"RoofMatl",
"Exterior1st",
"Exterior2nd",
"MasVnrType",
"Foundation",
"Heating",
"Electrical",
"GarageType",
"Fence",
"MiscFeature",
"SaleType",
"SaleCondition"

Y/N:
"CentralAir"

Ordinal 1-10:
"OverallQual",
"OverallCond"

Ordinal Po-Ex(5):
"ExterQual",
"ExterCond",
"HeatingQC",
"KitchenQual"

Ordinal Na-Ex(6):
"BsmtQual",
"BsmtCond",
"FireplaceQu",
"GarageQual",
"GarageCond"

Ordinal Na-Ex(5):
"PoolQC"

Ordinal Misc:
"BsmtExposure",
"BsmtFinType1",
"BsmtFinType2",
"Functional",
"GarageFinish",
"PavedDrive"??



In [210]:
#Ordinal Maps
datasets = [
  df_train,
  classification_df_train,
  df_test
]

for dataset in datasets:
  # Yes/No Map
  quality_map = {
    "N": 0,
    "Y": 1
  }

  dataset["CentralAir"] = dataset["CentralAir"].map(quality_map)

  # Poor - Excellent 5 Rankings Map
  quality_map = {
    "Po": 0,
    "NA": 0,
    "None": 0,
    "Fa": 1,
    "TA": 2,
    "Gd": 3,
    "Ex": 4
  }

  ordinal_po_ex_5 = [
  "ExterQual",
  "ExterCond",
  "HeatingQC",
  "KitchenQual"
  ]

  for category in ordinal_po_ex_5:
    dataset[category]=dataset[category].map(quality_map)

  # NA - Excellent 6 Rankings Map
  quality_map = {
    "NA": 0,
    "None": 0,
    "Po": 1,
    "Fa": 2,
    "TA": 3,
    "Gd": 4,
    "Ex": 5
  }

  ordinal_na_ex_6 = [
  "BsmtQual",
  "BsmtCond",
  "FireplaceQu",
  "GarageQual",
  "GarageCond"
  ]

  for category in ordinal_na_ex_6:
    dataset[category]=dataset[category].map(quality_map)

  # NA - Excellent 5 Rankings Map
  quality_map = {
    "NA": 0,
    "None": 0,
    "Fa": 1,
    "TA": 2,
    "Gd": 3,
    "Ex": 4
  }

  ordinal_na_ex_5 = [
    "PoolQC"
  ]

  for category in ordinal_na_ex_5:
    dataset[category]=dataset[category].map(quality_map)


  # Misc. Ordinal Rankings Map

  # "BsmtExposure"
  quality_map ={
    "NA": 0,
    "None": 0,
    "No": 0,
    "Mn":	2,
    "Av":	3,
    "Gd":	4
  }

  dataset["BsmtExposure"]=dataset["BsmtExposure"].map(quality_map)

  # "BsmtFinType1"
  # "BsmtFinType2"
  quality_map = {
    "NA": 0,
    "None": 0,
    "No": 0,
    "Unf": 1,
    "LwQ": 2,
    "Rec": 3,
    "BLQ": 4,
    "ALQ": 5,
    "GLQ": 6
  }

  dataset["BsmtFinType1"]=dataset["BsmtFinType1"].map(quality_map)
  dataset["BsmtFinType2"]=dataset["BsmtFinType2"].map(quality_map)

  # "Functional"
  quality_map ={
    "NA": 0,
    "None": 0,
    "Sal": 0,
    "Sev": 1,
    "Maj2": 2,
    "Maj1": 3,
    "Mod": 4,
    "Min2": 5,
    "Min1": 6,
    "Typ": 7
  }
  dataset["Functional"]=dataset["Functional"].map(quality_map)

  # "GarageFinish"
  quality_map ={
    "NA": 0,
    "None": 0,
    "Unf": 1,
    "RFn": 2,
    "Fin": 3
  }

  dataset["GarageFinish"]=dataset["GarageFinish"].map(quality_map)


  # "PavedDrive"
  quality_map ={
    "N": 0,
    "NA": 0,
    "None": 0,
    "P" : 1,
    "Y": 2
  }

  dataset["PavedDrive"]=dataset["PavedDrive"].map(quality_map)

In [211]:
# Regression dataset targeting sale price
X = df_train.drop(columns=["SalePrice"])
y=df_train["SalePrice"]



# Classification dataset targeting price category 
y_class = classification_df_train["PriceCategory"]
X_class = classification_df_train.drop(columns=["PriceCategory"])

X_test = df_test

In [212]:
#Nominal Maps
nominal_columns = [
"MSSubClass",
"MSZoning",
"Street",
"Alley",
"LotShape",
"LandContour",
"Utilities",
"LotConfig",
"LandSlope",
"Neighborhood",
"Condition1",
"Condition2",
"BldgType",
"HouseStyle",
"RoofStyle",
"RoofMatl",
"Exterior1st",
"Exterior2nd",
"MasVnrType",
"Foundation",
"Heating",
"Electrical",
"GarageType",
"Fence",
"MiscFeature",
"SaleType",
"SaleCondition"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat",
        OneHotEncoder(handle_unknown="ignore", drop="first"),
        nominal_columns)
    ],
    remainder="passthrough"
)

X = preprocessor.fit_transform(X)
X_class = preprocessor.transform(X_class)
X_test = preprocessor.transform(X_test)

c:\code\d789_task2\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [213]:
# Regression Split
X_train, X_val, y_train, y_val = train_test_split(
  X,
  y,
  test_size=0.2,
  random_state=42
)

# Classification Split
X_class_train, X_class_val, y_class_train, y_class_val = train_test_split(
  X_class,
  y_class,
  test_size=0.2,
  random_state=42
)


In [214]:
# Chi-Square Feature Selection - Classification Dataset - (Running before scaling to avoid negatives)
selector = SelectKBest(
  score_func=chi2,
  k=20
)

X_class_train_chi2 = selector.fit_transform(X_class_train, y_class_train)
X_class_val_chi2 = selector.transform(X_class_val)

In [215]:

# Scale regression dataset
reg_scaler = StandardScaler()
X_train = reg_scaler.fit_transform(X_train)
X_val = reg_scaler.transform(X_val)

# Scale classification dataset
class_scaler = StandardScaler()
X_class_train = class_scaler.fit_transform(X_class_train)
X_class_val = class_scaler.transform(X_class_val)

# Scale Chi-Square Classification Dataset
chi2_scaler = StandardScaler()
X_class_train_chi2 = chi2_scaler.fit_transform(X_class_train_chi2)
X_class_val_chi2 = chi2_scaler.transform(X_class_val_chi2)


# Fix classification dataset label type
y_class_train = y_class_train.astype("int32")
y_class_val = y_class_val.astype("int32")

In [216]:
# PCA Feature Selection - Regression DataSet
pca = PCA(n_components=0.95)

X_train_PCA = pca.fit_transform(X_train)
X_val_PCA = pca.transform(X_val)


In [217]:
# ANOVA F-Test Feature Selection - Regression & Classification Datasets
selector = SelectKBest(
  score_func=f_regression,
  k=20
)

X_train_ANOVA = selector.fit_transform(X_train, y_train)
X_val_ANOVA = selector.transform(X_val)

X_class_train_ANOVA = selector.fit_transform(X_class_train, y_class_train)
X_class_val_ANOVA = selector.transform(X_class_val)

In [218]:
# Regression Model - Linear Regression
model = LinearRegression()
results = []

# Full feature set
model.fit(X_train, y_train)

y_pred = model.predict(X_val)

mae = mean_absolute_error(
    y_val, 
    y_pred
)

rmse = root_mean_squared_error(
    y_val,
    y_pred,
)

r2 = r2_score(
    y_val,
    y_pred
)

results.append({
    "Model": "Linear Regression - Full Feature Set",
    "MAE": mae,
    "RMSE": rmse,
    "R2": r2
})  


# PCA Selected Feature Set
model.fit(X_train_PCA, y_train)

y_pred = model.predict(X_val_PCA)

mae = mean_absolute_error(
    y_val, 
    y_pred
)

rmse = root_mean_squared_error(
    y_val,
    y_pred,
)

r2 = r2_score(
    y_val,
    y_pred
)

results.append({
    "Model": "Linear Regression - PCA Feature Set",
    "MAE": mae,
    "RMSE": rmse,
    "R2": r2
})

# ANOVA Selected Feature Set
model.fit(X_train_ANOVA, y_train)

y_pred = model.predict(X_val_ANOVA)

mae = mean_absolute_error(
    y_val, 
    y_pred
)

rmse = root_mean_squared_error(
    y_val,
    y_pred,
)

r2 = r2_score(
    y_val,
    y_pred
)

results.append({
    "Model": "Linear Regression - PCA Feature Set",
    "MAE": mae,
    "RMSE": rmse,
    "R2": r2
})

results

[{'Model': 'Linear Regression - Full Feature Set',
  'MAE': 22096.547272557564,
  'RMSE': 51432.999832958274,
  'R2': 0.6551185177315693},
 {'Model': 'Linear Regression - PCA Feature Set',
  'MAE': 21709.602454151347,
  'RMSE': 34255.71922816183,
  'R2': 0.8470137686270598},
 {'Model': 'Linear Regression - PCA Feature Set',
  'MAE': 22908.69967917897,
  'RMSE': 36392.27067086933,
  'R2': 0.8273349361151224}]

In [219]:
# Classical Classification Model - Logistical Regression
model = LogisticRegression(
    max_iter=5000,
    random_state=42
)
results = []

model.fit(X_class_train, y_class_train)

y_pred = model.predict(X_class_val)

mae = mean_absolute_error(
    y_class_val, 
    y_pred
)

rmse = root_mean_squared_error(
    y_class_val,
    y_pred,
)

r2 = r2_score(
    y_class_val,
    y_pred
)

results.append({
    "Model": "Logistic Regression - Full Feature Set",
    "MAE": mae,
    "RMSE": rmse,
    "R2": r2
})

# ANOVA Selected Feature Set

model.fit(X_class_train_ANOVA, y_class_train)

y_pred = model.predict(X_class_val_ANOVA)

mae = mean_absolute_error(
    y_class_val, 
    y_pred
)

rmse = root_mean_squared_error(
    y_class_val,
    y_pred,
)

r2 = r2_score(
    y_class_val,
    y_pred
)

results.append({
    "Model": "Logistic Regression - ANOVA Feature Set",
    "MAE": mae,
    "RMSE": rmse,
    "R2": r2
})

# Chi-Squared Selected Feature Set
model.fit(X_class_train_chi2, y_class_train)

y_pred = model.predict(X_class_val_chi2)

mae = mean_absolute_error(
    y_class_val, 
    y_pred
)

rmse = root_mean_squared_error(
    y_class_val,
    y_pred,
)

r2 = r2_score(
    y_class_val,
    y_pred
)

results.append({
    "Model": "Logistic Regression - Chi-Squared Selected Feature Set",
    "MAE": mae,
    "RMSE": rmse,
    "R2": r2
})

results

[{'Model': 'Logistic Regression - Full Feature Set',
  'MAE': 0.20205479452054795,
  'RMSE': 0.44950505505561106,
  'R2': 0.7095360129484758},
 {'Model': 'Logistic Regression - ANOVA Feature Set',
  'MAE': 0.16095890410958905,
  'RMSE': 0.40119683960568414,
  'R2': 0.7686134340437011},
 {'Model': 'Logistic Regression - Chi-Squared Selected Feature Set',
  'MAE': 0.2226027397260274,
  'RMSE': 0.4861083931213425,
  'R2': 0.6603048287024549}]

In [220]:
results = []

# Regression ML Model - Full Feature Set
regressor = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(64, activation='relu'),
    Dense(1)
])

regressor.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

history = regressor.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2
)

results.append({
    "Model": "ML Regression - Full Feature Set",
    "loss": history.history['loss'][-1],
    "mae": history.history['mae'][-1],
    "val_loss": history.history['val_loss'][-1],
    "val_mae": history.history['val_mae'][-1]
})


# Regression ML Model - PCA Selected Feature Set
regressor = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_PCA.shape[1],)),
    Dense(64, activation='relu'),
    Dense(1)
])

regressor.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

history = regressor.fit(
    X_train_PCA,
    y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2
)

results.append({
    "Model": "ML Regression - PCA Selected Feature Set",
    "loss": history.history['loss'][-1],
    "mae": history.history['mae'][-1],
    "val_loss": history.history['val_loss'][-1],
    "val_mae": history.history['val_mae'][-1]
})

# Regression ML Model - ANOVA Selected Feature Set
regressor = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_ANOVA.shape[1],)),
    Dense(64, activation='relu'),
    Dense(1)
])

regressor.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

history = regressor.fit(
    X_train_ANOVA,
    y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2
)

results.append({
    "Model": "ML Regression - ANOVA Selected Feature Set",
    "loss": history.history['loss'][-1],
    "mae": history.history['mae'][-1],
    "val_loss": history.history['val_loss'][-1],
    "val_mae": history.history['val_mae'][-1]
})

results

Epoch 1/100


c:\code\d789_task2\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 39144767488.0000 - mae: 181521.0625 - val_loss: 37836693504.0000 - val_mae: 181088.8594
Epoch 2/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 39134097408.0000 - mae: 181497.1406 - val_loss: 37818359808.0000 - val_mae: 181048.6562
Epoch 3/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 39098007552.0000 - mae: 181422.1250 - val_loss: 37764530176.0000 - val_mae: 180934.6250
Epoch 4/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 39006281728.0000 - mae: 181235.2031 - val_loss: 37644206080.0000 - val_mae: 180681.4688
Epoch 5/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 38821720064.0000 - mae: 180859.9844 - val_loss: 37420318720.0000 - val_mae: 180209.9688
Epoch 6/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 38511497216.0000 - mae: 180224.6562 - val_loss: 37064732672.0000 - val_mae: 179458.0469
Epoch 7/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 38033592320.0000 - mae: 179231.7500 - val_loss: 36532817920.0000 - val_

KeyboardInterrupt: 

In [223]:
results = []

# Classification ML Model - Full Feature Set

model = Sequential([
    Dense(128, activation='relu', input_shape=(X_class_train.shape[1],)),
    Dense(64, activation='relu'),
    Dense(3, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_class_train,
    y_class_train.astype("int32"),
    epochs=100,
    batch_size=32,
    validation_split=0.2
)

results.append({
    "Model": "ML Classification - Full Feature Set",
    "accuracy": history.history['accuracy'][-1],
    "loss": history.history['loss'][-1],
    "val_accuracy": history.history['val_accuracy'][-1],
    "val_loss": history.history['val_loss'][-1]
})

# Classification ML Model - ANOVA Selected Feature Set
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_class_train_ANOVA.shape[1],)),
    Dense(64, activation='relu'),
    Dense(3, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_class_train_ANOVA,
    y_class_train.astype("int32"),
    epochs=100,
    batch_size=32,
    validation_split=0.2
)

results.append({
    "Model": "ML Classification - Full Feature Set",
    "accuracy": history.history['accuracy'][-1],
    "loss": history.history['loss'][-1],
    "val_accuracy": history.history['val_accuracy'][-1],
    "val_loss": history.history['val_loss'][-1]
})

# Classification ML Model - Chi-Squared Selected Feature Set
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_class_train_chi2.shape[1],)),
    Dense(64, activation='relu'),
    Dense(3, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_class_train_chi2,
    y_class_train.astype("int32"),
    epochs=100,
    batch_size=32,
    validation_split=0.2
)

results.append({
    "Model": "ML Classification - Full Feature Set",
    "accuracy": history.history['accuracy'][-1],
    "loss": history.history['loss'][-1],
    "val_accuracy": history.history['val_accuracy'][-1],
    "val_loss": history.history['val_loss'][-1]
})

results

Epoch 1/100


c:\code\d789_task2\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6424 - loss: 0.8055 - val_accuracy: 0.7393 - val_loss: 0.5956
Epoch 2/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8394 - loss: 0.4223 - val_accuracy: 0.7821 - val_loss: 0.4672
Epoch 3/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8972 - loss: 0.2933 - val_accuracy: 0.8077 - val_loss: 0.4318
Epoch 4/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9379 - loss: 0.2162 - val_accuracy: 0.8077 - val_loss: 0.4247
Epoch 5/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9550 - loss: 0.1680 - val_accuracy: 0.7906 - val_loss: 0.4295
Epoch 6/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9625 - loss: 0.1311 - val_accuracy: 0.7906 - val_loss: 0.4339
Epoch 7/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9722 - loss: 0.1013 - val_accuracy: 0.8120 - val_loss: 0.4461
Epoch 8/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9839 - loss: 0.0825 - val_accuracy: 0.7991 - val_loss: 0.4

c:\code\d789_task2\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6852 - loss: 0.7628 - val_accuracy: 0.7607 - val_loss: 0.5786
Epoch 2/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7677 - loss: 0.5274 - val_accuracy: 0.8162 - val_loss: 0.4809
Epoch 3/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8009 - loss: 0.4734 - val_accuracy: 0.8162 - val_loss: 0.4356
Epoch 4/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8137 - loss: 0.4468 - val_accuracy: 0.8162 - val_loss: 0.4279
Epoch 5/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8319 - loss: 0.4187 - val_accuracy: 0.8376 - val_loss: 0.4105
Epoch 6/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8373 - loss: 0.4015 - val_accuracy: 0.8291 - val_loss: 0.4109
Epoch 7/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8512 - loss: 0.3777 - val_accuracy: 0.8162 - val_loss: 0.4060
Epoch 8/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8490 - loss: 0.3717 - val_accuracy: 0.8462 - val_loss: 0.3

c:\code\d789_task2\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6060 - loss: 0.8507 - val_accuracy: 0.7137 - val_loss: 0.6962
Epoch 2/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7163 - loss: 0.6627 - val_accuracy: 0.7051 - val_loss: 0.6407
Epoch 3/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7388 - loss: 0.6047 - val_accuracy: 0.7350 - val_loss: 0.6207
Epoch 4/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7495 - loss: 0.5698 - val_accuracy: 0.7436 - val_loss: 0.6116
Epoch 5/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7687 - loss: 0.5394 - val_accuracy: 0.7479 - val_loss: 0.6199
Epoch 6/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7709 - loss: 0.5264 - val_accuracy: 0.7393 - val_loss: 0.6142
Epoch 7/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7859 - loss: 0.5007 - val_accuracy: 0.7308 - val_loss: 0.6378
Epoch 8/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7987 - loss: 0.4810 - val_accuracy: 0.7564 - val_loss: 0.6

[{'Model': 'ML Classification - Full Feature Set',
  'accuracy': 1.0,
  'loss': 0.00012735537893604487,
  'val_accuracy': 0.8205128312110901,
  'val_loss': 1.0911225080490112},
 {'Model': 'ML Classification - Full Feature Set',
  'accuracy': 0.9828693866729736,
  'loss': 0.048357151448726654,
  'val_accuracy': 0.7991452813148499,
  'val_loss': 0.6612192392349243},
 {'Model': 'ML Classification - Full Feature Set',
  'accuracy': 0.9860813617706299,
  'loss': 0.05937587842345238,
  'val_accuracy': 0.7564102411270142,
  'val_loss': 1.1553590297698975}]